# CITE-seq BMMC scIB bubble plot

## 1. Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 300
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]
mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["axes.grid"] = False
mpl.rcParams["legend.frameon"] = False
mpl.rcParams["xtick.major.size"] = 0
mpl.rcParams["ytick.major.size"] = 2

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS6_neurips2021_cite_bmmc_bubble_plot")
os.makedirs(_OUTDIR_ABS, exist_ok=True)


## 2. Paths : edit here

In [ ]:
BASE = "../.experiment_data/primary"

batch_key = "Site"

tools_with_hyper = {"NetworkVI", "MultiVI", "MultiMIL", "TotalVI"}

tool_paths = {
    "NetworkVI": [
        os.path.join(
            BASE,
            "experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False"
            "_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False"
            "_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0",
        ),
        os.path.join(
            BASE,
            "experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False"
            "_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False"
            "_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0",
        ),
    ],
    "MultiVI": os.path.join(
        BASE,
        "experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False"
        "_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0",
    ),
    "MultiMIL": os.path.join(
        BASE,
        "experiments/multimil_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False"
        "_scarches_False_ce_True_batcov_True_patcov_False_s0",
    ),
    "MIDAS": os.path.join(
        BASE,
        "experiments/midas_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False"
        "_scarches_False_batcov_False_patcov_False_s0",
    ),
    "WNN": os.path.join(
        BASE,
        "experiments/wnn_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False"
        "_scarches_False_s42",
    ),
    "MOFA+": os.path.join(
        BASE,
        "experiments/mofa_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False"
        "_scarches_False_s0",
    ),
    "TotalVI": os.path.join(
        BASE,
        "experiments/totalvi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False"
        "_scarches_False_batcov_False_patcov_False_s0",
    ),
}


## 3. Helper functions : find best hyper, load CSVs

In [ ]:
def get_best_hyper_path(base_path, batch_key):
    '''Mirror of get_best_hyper() in the original notebook.'''
    label_csvs = sorted(
        glob.glob(os.path.join(base_path, "hyper", "hyper*", "scib_results_label.csv"))
    )
    if not label_csvs:
        return None

    bio_scores = np.array([float(pd.read_csv(p)["Bio conservation"].iloc[0]) for p in label_csvs])

    if batch_key:
        batch_csvs = sorted(
            glob.glob(os.path.join(base_path, "hyper", "hyper*", f"scib_results_{batch_key}.csv"))
        )
        batch_scores = np.array(
            [float(pd.read_csv(p)["Batch correction"].iloc[0]) for p in batch_csvs]
        )
        combined = 0.6 * bio_scores + 0.4 * batch_scores
    else:
        combined = bio_scores

    best_idx = int(np.argmax(combined))
    return str(Path(label_csvs[best_idx]).parent)

def pick_best_folder(path_or_list, tool, batch_key, use_hyper):
    '''Given one path or a list of paths:'''
    paths = path_or_list if isinstance(path_or_list, list) else [path_or_list]

    if use_hyper:
        best_combined = -np.inf
        best_folder = None
        for p in paths:
            folder = get_best_hyper_path(p, batch_key)
            if folder is None:
                continue
            bio = float(
                pd.read_csv(os.path.join(folder, "scib_results_label.csv"))[
                    "Bio conservation"
                ].iloc[0]
            )
            if batch_key:
                bat = float(
                    pd.read_csv(os.path.join(folder, f"scib_results_{batch_key}.csv"))[
                        "Batch correction"
                    ].iloc[0]
                )
                comb = 0.6 * bio + 0.4 * bat
            else:
                comb = bio
            if comb > best_combined:
                best_combined = comb
                best_folder = folder
        if best_folder is not None:
            return best_folder
    return os.path.join(paths[0], "default")

BIO_METRICS = [
    "NMI cluster/label",
    "ARI cluster/label",
    "Cell type ASW",
    "Isolated labels",
    "Isolated label silhouette",
    "Graph cLISI",
    "HVG conservation",
    "Trajectory conservation",
]
BATCH_METRICS = [
    "PCR batch",
    "Batch ASW",
    "iLISI",
    "Graph connectivity",
    "kBET",
]

BIO_DISPLAY = [
    "NMI\ncluster/label",
    "ARI\ncluster/label",
    "Cell type\nASW",
    "Isolated\nlabel F1",
    "Isolated label\nsilhouette",
    "Graph cLISI",
    "HVG\nconservation",
    "Trajectory\nconservation",
]
BATCH_DISPLAY = [
    "PCR batch",
    "Batch ASW",
    "Graph iLISI",
    "Graph\nconnectivity",
    "kBET",
]

def load_metric_row(csv_path, metric_cols):
    '''Read a scib CSV and return {metric_name: float}.'''
    df = pd.read_csv(csv_path)
    row = df.iloc[0]
    result = {}
    for col in metric_cols:
        if col in row.index:
            result[col] = float(row[col])
        else:
            words = [w for w in col.split() if len(w) > 3]
            matched = [c for c in row.index if any(w.lower() in c.lower() for w in words)]
            result[col] = float(row[matched[0]]) if matched else np.nan
    return result


## 4. Load data for all tools

In [ ]:
tool_data = {}

for tool, path_or_list in tool_paths.items():
    use_hyper = tool in tools_with_hyper
    folder = pick_best_folder(path_or_list, tool, batch_key, use_hyper)
    print(f"{tool:12s}  →  {folder}")

    label_csv = os.path.join(folder, "scib_results_label.csv")
    bio_vals = load_metric_row(label_csv, BIO_METRICS)
    bio_agg = float(pd.read_csv(label_csv)["Bio conservation"].iloc[0])

    if batch_key:
        batch_csv = os.path.join(folder, f"scib_results_{batch_key}.csv")
        batch_vals = load_metric_row(batch_csv, BATCH_METRICS)
        batch_agg = float(pd.read_csv(batch_csv)["Batch correction"].iloc[0])
        combined = 0.6 * bio_agg + 0.4 * batch_agg
    else:
        batch_vals = {m: np.nan for m in BATCH_METRICS}
        batch_agg = np.nan
        combined = bio_agg

    tool_data[tool] = dict(
        bio=bio_vals,
        batch=batch_vals,
        bio_agg=bio_agg,
        batch_agg=batch_agg,
        combined=combined,
    )

tools_ordered = sorted(tool_data, key=lambda t: tool_data[t]["combined"], reverse=True)

print("\nRanking (paired_rate = 1.0):")
print(f"{'Tool':12s}  {'Combined':>9}  {'Bio':>9}  {'Batch':>9}")
print("-" * 45)
for t in tools_ordered:
    d = tool_data[t]
    print(f"{t:12s}  {d['combined']:9.4f}  {d['bio_agg']:9.4f}  {d['batch_agg']:9.4f}")


In [ ]:
for tool in tools_ordered:
    print(f"\n{tool}")
    print("  Bio:", {k: round(v, 4) for k, v in tool_data[tool]["bio"].items()})
    print("  Batch:", {k: round(v, 4) for k, v in tool_data[tool]["batch"].items()})


## 5. Ranking helpers

In [ ]:
n_tools = len(tools_ordered)

def rank_dict(value_dict, ascending=False):
    """Return {tool: rank}, rank 1 = best (highest score by default)."""
    vals = [(t, value_dict[t]) for t in tools_ordered]
    vals_sorted = sorted(vals, key=lambda x: (np.isnan(x[1]), x[1]), reverse=not ascending)
    return {t: r + 1 for r, (t, _) in enumerate(vals_sorted)}

overall_ranks = rank_dict({t: tool_data[t]["combined"] for t in tools_ordered})
bio_agg_ranks = rank_dict({t: tool_data[t]["bio_agg"] for t in tools_ordered})
batch_agg_ranks = rank_dict({t: tool_data[t]["batch_agg"] for t in tools_ordered})

bio_metric_ranks = [
    rank_dict({t: tool_data[t]["bio"][m] for t in tools_ordered}) for m in BIO_METRICS
]
batch_metric_ranks = [
    rank_dict({t: tool_data[t]["batch"][m] for t in tools_ordered}) for m in BATCH_METRICS
]


## 6. Colour maps & size mapping

In [ ]:
batch_cmap = LinearSegmentedColormap.from_list(
    "batch", ["#1a5276", "#2e86c1", "#85c1e9", "#d6eaf8", "#fafcff"], N=256
)
bio_cmap = LinearSegmentedColormap.from_list(
    "bio", ["#6c3483", "#a569bd", "#d2b4de", "#f5eef8", "#fdfeff"], N=256
)
overall_cmap = LinearSegmentedColormap.from_list(
    "overall", ["#145a32", "#27ae60", "#a9dfbf", "#eafaf1", "#fdfffe"], N=256
)

def rank_to_color(rank, cmap):
    """rank 1 → darkest, rank n_tools → lightest."""
    t = (rank - 1) / max(n_tools - 1, 1)
    return cmap(t)

MAX_S = 260
MIN_S = 4

def score_to_size(score):
    if np.isnan(score):
        return MIN_S
    return MIN_S + np.clip(score, 0, 1) * (MAX_S - MIN_S)


## 7. Draw bubble plot

In [ ]:
n_bio = len(BIO_METRICS)
n_batch = len(BATCH_METRICS)

X0 = 1.5
col = {}
col["overall_bar"] = X0 + 0
col["overall_bubble"] = X0 + 1
col["batch_bar"] = X0 + 2
for i in range(n_batch):
    col[f"batch_{i}"] = X0 + 3 + i
col["bio_bar"] = X0 + 3 + n_batch
for i in range(n_bio):
    col[f"bio_{i}"] = X0 + 4 + n_batch + i
n_cols_data = 4 + n_batch + n_bio
x_max = X0 + n_cols_data - 0.4

fig_w = 9.5
fig_h = 3.2 + 0.25 * max(0, n_tools - 7)
HEADER_Y = n_tools - 0.3

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_xlim(-0.1, x_max + 2.5)
ax.set_ylim(-0.8, n_tools + 1.8)
ax.axis("off")

row_y = {t: n_tools - 1 - i for i, t in enumerate(tools_ordered)}

def draw_header(ax, x0, x1, label, color, y=HEADER_Y, h=0.62):
    ax.add_patch(
        mpatches.FancyBboxPatch(
            (x0 - 0.45, y - 0.05),
            x1 - x0 + 0.9,
            h,
            boxstyle="round,pad=0.04",
            fc=color,
            ec="none",
            clip_on=False,
            zorder=3,
        )
    )
    ax.text(
        (x0 + x1) / 2,
        y + h / 2,
        label,
        ha="center",
        va="center",
        fontsize=7,
                color="white",
        clip_on=False,
        zorder=4,
    )

draw_header(ax, col["overall_bar"], col["overall_bubble"], "Overall", "#4a9ab5")
draw_header(ax, col["batch_bar"], col[f"batch_{n_batch-1}"], "Batch correction", "#5b9ec9")
draw_header(ax, col["bio_bar"], col[f"bio_{n_bio-1}"], "Bio conservation", "#c2588c")

for i, t in enumerate(tools_ordered):
    if i % 2 == 0:
        ax.add_patch(
            mpatches.Rectangle(
                (col["overall_bar"] - 0.45, row_y[t] - 0.5),
                x_max - (col["overall_bar"] - 0.45) + 0.1,
                1.0,
                fc="#f4f4f4",
                ec="none",
                zorder=0,
            )
        )

for t in tools_ordered:
    ax.text(
        col["overall_bar"] - 0.55,
        row_y[t],
        t,
        ha="right",
        va="center",
        fontsize=6.5,
                color="#c0392b" if t == "NetworkVI" else "black",
    )

LABEL_Y = HEADER_Y + 0.72
for i, name in enumerate(BATCH_DISPLAY):
    ax.text(
        col[f"batch_{i}"],
        LABEL_Y,
        name,
        ha="left",
        va="bottom",
        fontsize=4.8,
        rotation=45,
        rotation_mode="anchor",
        clip_on=False,
    )
for i, name in enumerate(BIO_DISPLAY):
    ax.text(
        col[f"bio_{i}"],
        LABEL_Y,
        name,
        ha="left",
        va="bottom",
        fontsize=4.8,
        rotation=45,
        rotation_mode="anchor",
        clip_on=False,
    )
for cx, label in [
    (col["overall_bubble"], "Overall\nscore"),
    (col["batch_bar"], "Overall\nscore"),
    (col["bio_bar"], "Overall\nscore"),
]:
    ax.text(cx, LABEL_Y, label, ha="center", va="bottom", fontsize=4.8, clip_on=False)

for sx in [col["batch_bar"] - 0.5, col["bio_bar"] - 0.5]:
    ax.plot([sx, sx], [-0.5, n_tools + 0.5], color="#bbbbbb", lw=0.8, zorder=1, clip_on=False)

def draw_bar(cx, tool, score, rank, cmap):
    y = row_y[tool]
    color = rank_to_color(rank, cmap)
    ax.add_patch(
        mpatches.Rectangle((cx - 0.45, y - 0.38), 0.9, 0.76, fc=color, ec="none", zorder=1)
    )
    txt_color = "white" if rank <= max(2, n_tools // 3) else "#333333"
    ax.text(
        cx, y, f"{score:.3f}", ha="center", va="center", fontsize=4.5, color=txt_color, zorder=2
    )

def draw_bubble(cx, tool, score, rank, cmap):
    y = row_y[tool]
    color = rank_to_color(rank, cmap)
    ax.scatter(
        cx, y, s=score_to_size(score), c=[color], linewidths=0.3, edgecolors="#666666", zorder=3
    )

for t in tools_ordered:
    draw_bar(col["overall_bar"], t, tool_data[t]["combined"], overall_ranks[t], overall_cmap)
    draw_bubble(col["overall_bubble"], t, tool_data[t]["combined"], overall_ranks[t], overall_cmap)

for t in tools_ordered:
    draw_bar(col["batch_bar"], t, tool_data[t]["batch_agg"], batch_agg_ranks[t], batch_cmap)
for i, m in enumerate(BATCH_METRICS):
    for t in tools_ordered:
        draw_bubble(
            col[f"batch_{i}"], t, tool_data[t]["batch"][m], batch_metric_ranks[i][t], batch_cmap
        )

for t in tools_ordered:
    draw_bar(col["bio_bar"], t, tool_data[t]["bio_agg"], bio_agg_ranks[t], bio_cmap)
for i, m in enumerate(BIO_METRICS):
    for t in tools_ordered:
        draw_bubble(col[f"bio_{i}"], t, tool_data[t]["bio"][m], bio_metric_ranks[i][t], bio_cmap)

leg_x = x_max + 0.35
leg_y0 = n_tools - 1.0

ax.text(
    leg_x + 0.1,
    leg_y0 + 0.72,
    "Ranking",
    ha="left",
    va="center",
    fontsize=6,
        clip_on=False,
)
seg_h = 0.72 / n_tools
for r in range(n_tools):
    ax.add_patch(
        mpatches.Rectangle(
            (leg_x, leg_y0 + 0.45 - r * seg_h),
            0.22,
            seg_h,
            fc=rank_to_color(r + 1, bio_cmap),
            ec="none",
            clip_on=False,
        )
    )
ax.text(leg_x + 0.25, leg_y0 + 0.45, "1", ha="left", va="top", fontsize=5, clip_on=False)
ax.text(
    leg_x + 0.25,
    leg_y0 + 0.45 - (n_tools - 1) * seg_h,
    str(n_tools),
    ha="left",
    va="bottom",
    fontsize=5,
    clip_on=False,
)

score_legend_entries = [(0.1, "10%"), (0.55, "55%"), (1.0, "100%")]
y_range = ax.get_ylim()[1] - ax.get_ylim()[0]

def bubble_radius_data(score_val):
    return (np.sqrt(score_to_size(score_val)) / 2.0 / 72.0) / fig_h * y_range

ax.text(
    leg_x + 0.1,
    leg_y0 - 1.0,
    "Score",
    ha="left",
    va="center",
    fontsize=6,
        clip_on=False,
)

gap = 0.10
yy = leg_y0 - 1.45
for score_val, label in score_legend_entries:
    r = bubble_radius_data(score_val)
    yy -= r
    ax.scatter(
        leg_x + 0.18,
        yy,
        s=score_to_size(score_val),
        c="#aaaaaa",
        linewidths=0.3,
        edgecolors="#555555",
        clip_on=False,
        zorder=5,
    )
    ax.text(leg_x + 0.42, yy, label, ha="left", va="center", fontsize=5, clip_on=False)
    yy -= r + gap

plt.tight_layout(pad=0.3)
plt.show()


## 8. Save

In [ ]:
out_base = "scib_bubble_plot_neurips2021_cite_bmmc_paired"
fig.savefig(os.path.join(_OUTDIR_ABS, f"{out_base}.pdf"), dpi=300, format="pdf", bbox_inches="tight", transparent=True)
fig.savefig(os.path.join(_OUTDIR_ABS, f"{out_base}.png"), dpi=300, format="png", bbox_inches="tight", transparent=True)
fig.savefig(os.path.join(_OUTDIR_ABS, f"{out_base}.svg"), format="svg", bbox_inches="tight", transparent=True)
print(f"Saved {out_base}.pdf / .png / .svg")
